# 01 — Run Analysis & Save Results

This notebook runs all analyses for the BGA-AISNP project and saves intermediate results
(numpy arrays, CSVs) to `results/analysis_arrays/` for visualization in Notebook 02.

**Analyses:**
1. Dimensionality Reduction (PCA, t-SNE, UMAP, popVAE latent)
2. AISNP Selection & Feature Importance (Top 20)
3. Allele Frequency Distributions
4. Consolidated Benchmark (all 17+ models)
5. FREEFORM With/Without Comparison
6. Learning Curves (Generative vs Discriminative)
7. Timing & Latency Measurement

## 0. Setup & Dependencies

In [1]:
# Install missing packages (run once)
%pip install torch xgboost lightgbm catboost ngboost umap-learn tabpfn openpyxl --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, balanced_accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.path.abspath("")).parent
sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "results" / "analysis_arrays"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")

Project root: c:\Users\Admin\Uni\BGA-AISNP-NEW
Output dir:   c:\Users\Admin\Uni\BGA-AISNP-NEW\results\analysis_arrays


## 1. Load Data

In [3]:
from src.preprocessing import load_continental, load_eas

# Tier 1: Continental (5 super-populations)
X_cont, y_super, y_pop_cont, le_dict, snp_ids, df_meta = load_continental()
# Tier 2: East-Asian (5 sub-populations)
X_eas, y_super_eas, y_pop_eas, le_dict_eas, snp_ids_eas, df_meta_eas = load_eas()

print(f"Continental: X={X_cont.shape}, classes={list(le_dict['super_pop'].classes_)}")
print(f"EAS:         X={X_eas.shape}, classes={list(le_dict_eas['pop'].classes_)}")

# Save raw data arrays
np.save(OUTPUT_DIR / "X_cont.npy", X_cont)
np.save(OUTPUT_DIR / "y_super.npy", y_super)
np.save(OUTPUT_DIR / "X_eas.npy", X_eas)
np.save(OUTPUT_DIR / "y_pop_eas.npy", y_pop_eas)
np.savez(OUTPUT_DIR / "label_info.npz",
         super_pop_classes=le_dict["super_pop"].classes_,
         pop_cont_classes=le_dict["pop"].classes_,
         eas_pop_classes=le_dict_eas["pop"].classes_,
         snp_ids=np.array(snp_ids, dtype=object),
         snp_ids_eas=np.array(snp_ids_eas, dtype=object))
df_meta.to_csv(OUTPUT_DIR / "df_meta_cont.csv", index=False)
df_meta_eas.to_csv(OUTPUT_DIR / "df_meta_eas.csv", index=False)
print("Saved raw data arrays.")

[preprocess] Loaded AISNP_by_sample_continental.csv  ->  2504 samples, 51 columns
[preprocess] Found 24 SNPs
[preprocess] super_pop classes: ['AFR', 'AMR', 'EAS', 'EUR', 'SAS']
[preprocess] pop       classes: ['ACB', 'ASW', 'BEB', 'CDX', 'CEU', 'CHB', 'CHS', 'CLM', 'ESN', 'FIN', 'GBR', 'GIH', 'GWD', 'IBS', 'ITU', 'JPT', 'KHV', 'LWK', 'MSL', 'MXL', 'PEL', 'PJL', 'PUR', 'STU', 'TSI', 'YRI']
[preprocess] Loaded AISNP_by_sample_EAS_only.csv  ->  504 samples, 71 columns
[preprocess] Found 34 SNPs
[preprocess] super_pop classes: ['EAS']
[preprocess] pop       classes: ['CDX', 'CHB', 'CHS', 'JPT', 'KHV']
Continental: X=(2504, 24), classes=['AFR', 'AMR', 'EAS', 'EUR', 'SAS']
EAS:         X=(504, 34), classes=['CDX', 'CHB', 'CHS', 'JPT', 'KHV']
Saved raw data arrays.


## 2. Dimensionality Reduction — Continental (Tier 1)
PCA, t-SNE, UMAP on the 2,504 continental samples.

In [4]:
import umap

scaler_cont = StandardScaler()
X_cont_scaled = scaler_cont.fit_transform(X_cont)

# --- PCA ---
print("Running PCA (Continental)...")
pca_cont = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_cont = pca_cont.fit_transform(X_cont_scaled)
pca_var_cont = pca_cont.explained_variance_ratio_
print(f"  PCA explained variance: PC1={pca_var_cont[0]:.3f}, PC2={pca_var_cont[1]:.3f}")

# --- t-SNE ---
print("Running t-SNE (Continental)...")
tsne_cont = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30, max_iter=1000)
X_tsne_cont = tsne_cont.fit_transform(X_cont_scaled)

# --- UMAP ---
print("Running UMAP (Continental)...")
umap_cont = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
X_umap_cont = umap_cont.fit_transform(X_cont_scaled)

# Save
np.save(OUTPUT_DIR / "dimred_pca_cont.npy", X_pca_cont)
np.save(OUTPUT_DIR / "dimred_pca_cont_var.npy", pca_var_cont)
np.save(OUTPUT_DIR / "dimred_tsne_cont.npy", X_tsne_cont)
np.save(OUTPUT_DIR / "dimred_umap_cont.npy", X_umap_cont)
print("Saved Continental dim-reduction arrays.")

Running PCA (Continental)...
  PCA explained variance: PC1=0.287, PC2=0.229
Running t-SNE (Continental)...
Running UMAP (Continental)...
Saved Continental dim-reduction arrays.


## 3. Dimensionality Reduction — EAS (Tier 2)
PCA, t-SNE, UMAP on the 504 East-Asian samples.

In [5]:
scaler_eas = StandardScaler()
X_eas_scaled = scaler_eas.fit_transform(X_eas)

# --- PCA ---
print("Running PCA (EAS)...")
pca_eas = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_eas = pca_eas.fit_transform(X_eas_scaled)
pca_var_eas = pca_eas.explained_variance_ratio_
print(f"  PCA explained variance: PC1={pca_var_eas[0]:.3f}, PC2={pca_var_eas[1]:.3f}")

# --- t-SNE ---
print("Running t-SNE (EAS)...")
tsne_eas = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30, max_iter=1000)
X_tsne_eas = tsne_eas.fit_transform(X_eas_scaled)

# --- UMAP ---
print("Running UMAP (EAS)...")
umap_eas = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
X_umap_eas = umap_eas.fit_transform(X_eas_scaled)

# Save
np.save(OUTPUT_DIR / "dimred_pca_eas.npy", X_pca_eas)
np.save(OUTPUT_DIR / "dimred_pca_eas_var.npy", pca_var_eas)
np.save(OUTPUT_DIR / "dimred_tsne_eas.npy", X_tsne_eas)
np.save(OUTPUT_DIR / "dimred_umap_eas.npy", X_umap_eas)
print("Saved EAS dim-reduction arrays.")

Running PCA (EAS)...
  PCA explained variance: PC1=0.164, PC2=0.102
Running t-SNE (EAS)...


Running UMAP (EAS)...
Saved EAS dim-reduction arrays.


## 4. popVAE Latent Space
Train popVAE on both tiers and extract 10-D latent representations, then PCA to 2-D for visualization.

In [6]:
from src.popvae_model import PopVAEClassifier

# --- Continental popVAE ---
print("Training popVAE (Continental)... this may take a few minutes.")
popvae_cont = PopVAEClassifier(
    latent_dim=10, enc_hidden=(128, 64),
    beta=1.0, gamma=10.0, epochs=200, patience=20,
    random_state=RANDOM_STATE, device="cpu"
)
popvae_cont.fit(X_cont, y_super)
Z_cont = popvae_cont.encode_latent(X_cont)  # (2504, 10)
Z_cont_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Z_cont)
print(f"  Continental latent shape: {Z_cont.shape} -> 2D: {Z_cont_2d.shape}")

np.save(OUTPUT_DIR / "dimred_popvae_cont.npy", Z_cont)
np.save(OUTPUT_DIR / "dimred_popvae_cont_2d.npy", Z_cont_2d)

# --- EAS popVAE ---
print("Training popVAE (EAS)...")
popvae_eas = PopVAEClassifier(
    latent_dim=10, enc_hidden=(128, 64),
    beta=1.0, gamma=10.0, epochs=200, patience=20,
    random_state=RANDOM_STATE, device="cpu"
)
popvae_eas.fit(X_eas, y_pop_eas)
Z_eas = popvae_eas.encode_latent(X_eas)  # (504, 10)
Z_eas_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Z_eas)
print(f"  EAS latent shape: {Z_eas.shape} -> 2D: {Z_eas_2d.shape}")

np.save(OUTPUT_DIR / "dimred_popvae_eas.npy", Z_eas)
np.save(OUTPUT_DIR / "dimred_popvae_eas_2d.npy", Z_eas_2d)
print("Saved popVAE latent arrays.")

Training popVAE (Continental)... this may take a few minutes.
  Continental latent shape: (2504, 10) -> 2D: (2504, 2)
Training popVAE (EAS)...
  EAS latent shape: (504, 10) -> 2D: (504, 2)
Saved popVAE latent arrays.


## 5. Feature Importance & Top 20 AISNP Ranking
Average feature importance from RF, GradientBoosting, XGBoost → rank top 20 AISNPs per tier.

In [7]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

def compute_feature_importances(X, y, snp_names, tier_name):
    """Train 3 tree models and average their feature importances."""
    models = {
        "RandomForest": RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=300, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1,
            verbosity=0, eval_metric="mlogloss"),
    }
    importances = {}
    for name, model in models.items():
        print(f"  Fitting {name} ({tier_name})...")
        model.fit(X, y)
        importances[name] = model.feature_importances_

    avg_imp = np.mean(list(importances.values()), axis=0)
    ranking = np.argsort(avg_imp)[::-1]
    top20_idx = ranking[:20]
    top20_snps = np.array([snp_names[i] for i in top20_idx], dtype=object)
    top20_scores = avg_imp[top20_idx]

    np.savez(
        OUTPUT_DIR / f"feature_importance_{tier_name}.npz",
        importances_rf=importances["RandomForest"],
        importances_gb=importances["GradientBoosting"],
        importances_xgb=importances["XGBoost"],
        avg_importance=avg_imp,
        top20_idx=top20_idx,
        top20_snps=top20_snps,
        top20_scores=top20_scores,
        snp_ids=np.array(snp_names, dtype=object),
    )
    print(f"  Top 5 AISNPs ({tier_name}): {list(top20_snps[:5])}")
    return top20_snps, top20_scores, importances

print("=== Tier 1 (Continental) ===")
top20_cont, scores_cont, imp_cont = compute_feature_importances(
    X_cont, y_super, snp_ids, "tier1")

print("=== Tier 2 (EAS) ===")
top20_eas, scores_eas, imp_eas = compute_feature_importances(
    X_eas, y_pop_eas, snp_ids_eas, "tier2")

print("Saved feature importance arrays.")

=== Tier 1 (Continental) ===
  Fitting RandomForest (tier1)...
  Fitting GradientBoosting (tier1)...
  Fitting XGBoost (tier1)...
  Top 5 AISNPs (tier1): ['rs1229984', 'rs10497191', 'rs28777', 'rs7554936', 'rs513265']
=== Tier 2 (EAS) ===
  Fitting RandomForest (tier2)...
  Fitting GradientBoosting (tier2)...
  Fitting XGBoost (tier2)...
  Top 5 AISNPs (tier2): ['rs79200067', 'rs11629323', 'rs464165', 'rs434124', 'rs546642722']
Saved feature importance arrays.


## 6. Allele Frequency Distributions
Compute minor allele frequency per population for the top 20 AISNPs.

In [8]:
def compute_allele_frequencies(X, y, snp_names, le, top20_snps):
    """Compute mean allele frequency (0-1) per population for top 20 SNPs."""
    classes = le.classes_
    snp_to_col = {s: i for i, s in enumerate(snp_names)}
    rows = []
    for snp in top20_snps:
        col_idx = snp_to_col.get(snp)
        if col_idx is None:
            continue
        row = {"SNP": snp}
        for cls_idx, cls_name in enumerate(classes):
            mask = (y == cls_idx)
            vals = X[mask, col_idx]
            row[cls_name] = float(np.nanmean(vals) / 2.0)  # normalize 0-2 → 0-1
        rows.append(row)
    return pd.DataFrame(rows)

# Tier 1
af_tier1 = compute_allele_frequencies(
    X_cont, y_super, snp_ids, le_dict["super_pop"], top20_cont)
af_tier1.to_csv(OUTPUT_DIR / "allele_freq_tier1_top20.csv", index=False)
print("Tier 1 allele frequencies (top 5 SNPs):")
print(af_tier1.head().to_string(index=False))

# Tier 2
af_tier2 = compute_allele_frequencies(
    X_eas, y_pop_eas, snp_ids_eas, le_dict_eas["pop"], top20_eas)
af_tier2.to_csv(OUTPUT_DIR / "allele_freq_tier2_top20.csv", index=False)
print("Tier 2 allele frequencies (top 5 SNPs):")
print(af_tier2.head().to_string(index=False))
print("Saved allele frequency CSVs.")

Tier 1 allele frequencies (top 5 SNPs):
       SNP      AFR      AMR      EAS      EUR      SAS
 rs1229984 0.001513 0.057637 0.697421 0.028827 0.020450
rs10497191 0.922088 0.185879 0.096230 0.123260 0.097137
   rs28777 0.195159 0.492795 0.117063 0.956262 0.222904
 rs7554936 0.969743 0.319885 0.143849 0.333996 0.218814
  rs513265 0.338124 0.077810 0.012897 0.019881 0.464213
Tier 2 allele frequencies (top 5 SNPs):
        SNP      CDX      CHB      CHS      JPT      KHV
 rs79200067 0.000000 0.000000 0.000000 0.139423 0.000000
 rs11629323 0.091398 0.597087 0.404762 0.812500 0.242424
   rs464165 0.000000 0.000000 0.000000 0.100962 0.000000
   rs434124 0.795699 0.165049 0.376190 0.096154 0.641414
rs546642722 0.322581 0.009709 0.009524 0.000000 0.176768
Saved allele frequency CSVs.


## 7. Consolidate All Benchmark Results
Merge results from 3 sources into a unified benchmark CSV covering all 17+ models.

In [9]:
import openpyxl

# --- Source 1: Base models (ALL_model_comparison.csv) ---
df_base = pd.read_csv(PROJECT_ROOT / "results" / "ALL_model_comparison.csv")
df_base["tier"] = df_base["dataset"].map({
    "stage1_super_pop": "tier1", "stage2_EAS_pop": "tier2"})
df_base_norm = df_base[["tier", "model", "test_accuracy", "test_balanced_acc",
    "test_f1_macro", "test_roc_auc_macro", "fit_time_sec"]].copy()
df_base_norm.columns = ["tier", "model", "accuracy", "balanced_acc",
    "f1_macro", "roc_auc", "fit_time_sec"]
df_base_norm["mcc"] = float("nan")
df_base_norm["source"] = "base_pipeline"

# --- Source 2: Classic models ---
df_classic = pd.read_csv(PROJECT_ROOT / "results" / "classic_models" / "classic_models_results.csv")
df_classic["tier"] = df_classic["dataset"].map({
    "Continental (super_pop)": "tier1", "EastAsia (pop)": "tier2"})
df_classic_norm = df_classic[["tier", "method", "accuracy", "balanced_accuracy",
    "mcc", "f1_macro", "roc_auc_macro", "fit_time_sec"]].copy()
df_classic_norm.columns = ["tier", "model", "accuracy", "balanced_acc",
    "mcc", "f1_macro", "roc_auc", "fit_time_sec"]
df_classic_norm["source"] = "classic_models"

# --- Source 3: Aggregated results (Excel) ---
df_agg = pd.read_excel(
    PROJECT_ROOT / "reports" / "aggregated_results" / "model_metrics.xlsx")
df_agg = df_agg.dropna(subset=["dataset"])
df_agg["tier"] = df_agg["dataset"].map({"Continental": "tier1", "East Asia": "tier2"})
df_agg_norm = df_agg[["tier", "method", "accuracy", "mcc"]].copy()
df_agg_norm.columns = ["tier", "model", "accuracy", "mcc"]
df_agg_norm["balanced_acc"] = np.nan
df_agg_norm["f1_macro"] = np.nan
df_agg_norm["roc_auc"] = np.nan
df_agg_norm["fit_time_sec"] = np.nan
df_agg_norm["source"] = "aggregated_excel"

# --- Merge all, prefer base > classic > aggregated ---
df_all = pd.concat([df_base_norm, df_classic_norm, df_agg_norm], ignore_index=True)

# Convert roc_auc "N/A" strings to NaN
df_all["roc_auc"] = pd.to_numeric(df_all["roc_auc"], errors="coerce")
df_all["mcc"] = pd.to_numeric(df_all["mcc"], errors="coerce")

# Deduplicate: keep first occurrence (base_pipeline > classic_models > aggregated_excel)
df_all = df_all.drop_duplicates(subset=["tier", "model"], keep="first")
df_all = df_all.sort_values(["tier", "accuracy"], ascending=[True, False]).reset_index(drop=True)

# Categorize models
model_categories = {
    "LogisticRegression": "Traditional ML",
    "RandomForest": "Traditional ML",
    "GradientBoosting": "Boosting",
    "XGBoost": "Boosting",
    "LightGBM": "Boosting",
    "CatBoost": "Boosting",
    "NGBoost": "Boosting",
    "GA-SVM": "Traditional ML",
    "GenerativeBGA": "Bayesian",
    "popVAE": "Deep Learning",
    "SVD-MLP-Adv": "Deep Learning",
    "DietNetworks": "Deep Learning",
    "FederatedMLP": "Deep Learning",
    "FREEFORM": "Ensemble",
    "TabPFN": "Deep Learning",
    "MLP-Geo": "Deep Learning",
}
df_all["category"] = df_all["model"].map(model_categories).fillna("Other")

df_all.to_csv(OUTPUT_DIR / "benchmark_all_models.csv", index=False)
print(f"Consolidated benchmark: {len(df_all)} rows, {df_all['model'].nunique()} unique models")
print(f"Tier 1 models ({(df_all['tier']=='tier1').sum()}):") 
print(df_all[df_all["tier"]=="tier1"][["model","accuracy","mcc","category"]].to_string(index=False))
print(f"Tier 2 models ({(df_all['tier']=='tier2').sum()}):") 
print(df_all[df_all["tier"]=="tier2"][["model","accuracy","mcc","category"]].to_string(index=False))

Consolidated benchmark: 32 rows, 16 unique models
Tier 1 models (16):
             model  accuracy      mcc       category
LogisticRegression  0.974100      NaN Traditional ML
            TabPFN  0.974052 0.967277  Deep Learning
       SVD-MLP-Adv  0.970060 0.962291  Deep Learning
            popVAE  0.970060 0.962219  Deep Learning
  GradientBoosting  0.968100      NaN       Boosting
          LightGBM  0.968100      NaN       Boosting
      FederatedMLP  0.968064 0.959796  Deep Learning
           XGBoost  0.966100      NaN       Boosting
     GenerativeBGA  0.966068 0.957249       Bayesian
      RandomForest  0.962100      NaN Traditional ML
          CatBoost  0.962100      NaN       Boosting
          FREEFORM  0.960080 0.949809       Ensemble
            GA-SVM  0.954092 0.942062 Traditional ML
           MLP-Geo  0.936128 0.919398  Deep Learning
           NGBoost  0.936100      NaN       Boosting
      DietNetworks  0.928144 0.910007  Deep Learning
Tier 2 models (16):
         

## 8. FREEFORM With/Without Comparison
Compare base models trained on raw features vs FREEFORM-engineered features, plus the full stacking ensemble.

In [10]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from src.freeform_model import SNPFeatureEngineer, FreeformBGAClassifier

def run_freeform_comparison(X, y, tier_name):
    """Compare models with and without FREEFORM feature engineering."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

    base_models = {
        "XGBoost": XGBClassifier(
            n_estimators=300, random_state=RANDOM_STATE, verbosity=0,
            eval_metric="mlogloss", n_jobs=-1),
        "LightGBM": LGBMClassifier(
            n_estimators=300, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
    }

    results = []

    # --- Without FREEFORM (raw features) ---
    for name, model in base_models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        results.append({
            "model": name, "condition": "Raw Features",
            "accuracy": accuracy_score(y_test, y_pred),
            "mcc": matthews_corrcoef(y_test, y_pred),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        })
        print(f"  {name} (raw):     acc={results[-1]['accuracy']:.4f}  mcc={results[-1]['mcc']:.4f}")

    # --- With FREEFORM feature engineering ---
    eng = SNPFeatureEngineer(top_k=15, n_interaction_pairs=45)
    eng.fit(X_train, y_train)
    X_train_eng = eng.transform(X_train)
    X_test_eng = eng.transform(X_test)

    # Impute + scale the engineered features
    imp = SimpleImputer(strategy="median")
    X_train_eng = imp.fit_transform(X_train_eng)
    X_test_eng = imp.transform(X_test_eng)
    sc = StandardScaler()
    X_train_eng_sc = sc.fit_transform(X_train_eng)
    X_test_eng_sc = sc.transform(X_test_eng)

    base_models_eng = {
        "XGBoost": XGBClassifier(
            n_estimators=300, random_state=RANDOM_STATE, verbosity=0,
            eval_metric="mlogloss", n_jobs=-1),
        "LightGBM": LGBMClassifier(
            n_estimators=300, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1),
        "LogisticRegression": LogisticRegression(
            max_iter=2000, random_state=RANDOM_STATE),
    }

    for name, model in base_models_eng.items():
        X_tr = X_train_eng_sc if name == "LogisticRegression" else X_train_eng
        X_te = X_test_eng_sc if name == "LogisticRegression" else X_test_eng
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        results.append({
            "model": name, "condition": "FREEFORM Features",
            "accuracy": accuracy_score(y_test, y_pred),
            "mcc": matthews_corrcoef(y_test, y_pred),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        })
        print(f"  {name} (FREEFORM): acc={results[-1]['accuracy']:.4f}  mcc={results[-1]['mcc']:.4f}")

    # --- Full FREEFORM Stacking ---
    freeform = FreeformBGAClassifier(top_k=15, n_interaction_pairs=45)
    freeform.fit(X_train, y_train)
    y_pred = freeform.predict(X_test)
    results.append({
        "model": "FREEFORM (Stacking)", "condition": "Full Stacking",
        "accuracy": accuracy_score(y_test, y_pred),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    })
    print(f"  FREEFORM Stacking: acc={results[-1]['accuracy']:.4f}  mcc={results[-1]['mcc']:.4f}")

    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_DIR / f"freeform_comparison_{tier_name}.csv", index=False)
    return df

print("=== Tier 1 (Continental) ===")
df_ff_t1 = run_freeform_comparison(X_cont, y_super, "tier1")

print("=== Tier 2 (EAS) ===")
df_ff_t2 = run_freeform_comparison(X_eas, y_pop_eas, "tier2")
print("Saved FREEFORM comparison CSVs.")

=== Tier 1 (Continental) ===
  XGBoost (raw):     acc=0.9581  mcc=0.9471
  LightGBM (raw):     acc=0.9681  mcc=0.9597
  LogisticRegression (raw):     acc=0.9701  mcc=0.9622
  XGBoost (FREEFORM): acc=0.9641  mcc=0.9547
  LightGBM (FREEFORM): acc=0.9661  mcc=0.9572
  LogisticRegression (FREEFORM): acc=0.9581  mcc=0.9471
  FREEFORM Stacking: acc=0.9601  mcc=0.9498
=== Tier 2 (EAS) ===
  XGBoost (raw):     acc=0.6733  mcc=0.5945
  LightGBM (raw):     acc=0.7030  mcc=0.6324
  LogisticRegression (raw):     acc=0.7327  mcc=0.6717
  XGBoost (FREEFORM): acc=0.7030  mcc=0.6313
  LightGBM (FREEFORM): acc=0.6931  mcc=0.6193
  LogisticRegression (FREEFORM): acc=0.6733  mcc=0.5974
  FREEFORM Stacking: acc=0.7426  mcc=0.6821
Saved FREEFORM comparison CSVs.


## 9. Learning Curves — Generative vs Discriminative
Compare stability of Bayesian generative (Naive Bayes) vs discriminative (XGBoost) across training set sizes.

In [11]:
from src.generative_model import GenerativeBGAModel

def run_learning_curves(X, y, snp_names, le, tier_name, n_splits=5):
    """Learning curves for XGBoost vs GenerativeBGA."""
    train_sizes_frac = np.linspace(0.1, 1.0, 10)
    y_encoded = y.copy()
    num_classes = len(le.classes_)

    sss = StratifiedShuffleSplit(n_splits=n_splits, test_size=0.2, random_state=RANDOM_STATE)
    results = {"XGBoost": {"sizes": [], "train_mean": [], "train_std": [],
                           "test_mean": [], "test_std": []},
               "Generative": {"sizes": [], "train_mean": [], "train_std": [],
                              "test_mean": [], "test_std": []}}

    for frac in train_sizes_frac:
        scores = {"XGBoost": {"train": [], "test": []},
                  "Generative": {"train": [], "test": []}}

        for train_idx, test_idx in sss.split(X, y_encoded):
            n_train = max(int(len(train_idx) * frac), num_classes)
            train_sub = train_idx[:n_train]

            X_tr, y_tr = X[train_sub], y_encoded[train_sub]
            X_te, y_te = X[test_idx], y_encoded[test_idx]

            # XGBoost
            xgb = XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                verbosity=0, eval_metric="mlogloss", n_jobs=-1)
            xgb.fit(X_tr, y_tr)
            scores["XGBoost"]["train"].append(xgb.score(X_tr, y_tr))
            scores["XGBoost"]["test"].append(xgb.score(X_te, y_te))

            # Generative
            gen = GenerativeBGAModel(smoothing_alpha=1.0)
            y_tr_str = le.inverse_transform(y_tr)
            gen.fit(X_tr, y_tr_str, snp_names)
            # predict returns strings, need to compare with strings
            y_te_str = le.inverse_transform(y_te)
            gen_tr_pred = gen.predict(X_tr)
            gen_te_pred = gen.predict(X_te)
            scores["Generative"]["train"].append(np.mean(gen_tr_pred == le.inverse_transform(y_tr)))
            scores["Generative"]["test"].append(np.mean(gen_te_pred == y_te_str))

        n_samples = int(len(train_idx) * frac)
        for name in results:
            results[name]["sizes"].append(n_samples)
            results[name]["train_mean"].append(np.mean(scores[name]["train"]))
            results[name]["train_std"].append(np.std(scores[name]["train"]))
            results[name]["test_mean"].append(np.mean(scores[name]["test"]))
            results[name]["test_std"].append(np.std(scores[name]["test"]))

        print(f"  {tier_name} frac={frac:.1f}: XGB_test={results['XGBoost']['test_mean'][-1]:.4f} "
              f"Gen_test={results['Generative']['test_mean'][-1]:.4f}")

    # Flatten and save
    save_dict = {}
    for model_name, data in results.items():
        for key, vals in data.items():
            save_dict[f"{model_name}_{key}"] = np.array(vals)
    np.savez(OUTPUT_DIR / f"learning_curves_{tier_name}.npz", **save_dict)
    return results

print("=== Tier 1 (Continental) ===")
lc_t1 = run_learning_curves(X_cont, y_super, snp_ids, le_dict["super_pop"], "tier1")

print("=== Tier 2 (EAS) ===")
lc_t2 = run_learning_curves(X_eas, y_pop_eas, snp_ids_eas, le_dict_eas["pop"], "tier2")
print("Saved learning curve arrays.")

=== Tier 1 (Continental) ===
  tier1 frac=0.1: XGB_test=0.9102 Gen_test=0.9441
  tier1 frac=0.2: XGB_test=0.9246 Gen_test=0.9505
  tier1 frac=0.3: XGB_test=0.9305 Gen_test=0.9509
  tier1 frac=0.4: XGB_test=0.9349 Gen_test=0.9513
  tier1 frac=0.5: XGB_test=0.9397 Gen_test=0.9525
  tier1 frac=0.6: XGB_test=0.9429 Gen_test=0.9533
  tier1 frac=0.7: XGB_test=0.9393 Gen_test=0.9529
  tier1 frac=0.8: XGB_test=0.9449 Gen_test=0.9525
  tier1 frac=0.9: XGB_test=0.9469 Gen_test=0.9525
  tier1 frac=1.0: XGB_test=0.9485 Gen_test=0.9525
=== Tier 2 (EAS) ===
  tier2 frac=0.1: XGB_test=0.4792 Gen_test=0.6376
  tier2 frac=0.2: XGB_test=0.5644 Gen_test=0.6752
  tier2 frac=0.3: XGB_test=0.5822 Gen_test=0.7188
  tier2 frac=0.4: XGB_test=0.6198 Gen_test=0.7089
  tier2 frac=0.5: XGB_test=0.6337 Gen_test=0.7089
  tier2 frac=0.6: XGB_test=0.6337 Gen_test=0.7188
  tier2 frac=0.7: XGB_test=0.6535 Gen_test=0.7347
  tier2 frac=0.8: XGB_test=0.6495 Gen_test=0.7248
  tier2 frac=0.9: XGB_test=0.6594 Gen_test=0.7188


## 10. Timing & Latency Measurement
Measure training time (no tuning) and inference latency for all models.

In [12]:
from catboost import CatBoostClassifier
from ngboost import NGBClassifier
from src.tabpfn_model import make_tabpfn_classifier
from src.mlp_geo_model import MLPGeoModel

def measure_timing(X, y, snp_names, le, tier_name):
    """Measure train time and inference latency for all models."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    n_classes = len(np.unique(y_train))
    results = []

    # Define all models to benchmark
    model_specs = [
        ("LogisticRegression", lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])),
        ("RandomForest", lambda: RandomForestClassifier(
            n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1)),
        ("GradientBoosting", lambda: GradientBoostingClassifier(
            n_estimators=300, random_state=RANDOM_STATE)),
        ("XGBoost", lambda: XGBClassifier(
            n_estimators=300, random_state=RANDOM_STATE, verbosity=0,
            eval_metric="mlogloss", n_jobs=-1)),
        ("LightGBM", lambda: LGBMClassifier(
            n_estimators=300, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)),
        ("CatBoost", lambda: CatBoostClassifier(
            iterations=300, random_state=RANDOM_STATE, verbose=0)),
        ("NGBoost", lambda: NGBClassifier(
            n_estimators=200, random_state=RANDOM_STATE, verbose=False)),
    ]

    for model_name, model_fn in model_specs:
        print(f"  Timing {model_name} ({tier_name})...")
        try:
            model = model_fn()
            t0 = time.time()
            model.fit(X_train, y_train)
            train_time = time.time() - t0

            inf_times = []
            for _ in range(10):
                t0 = time.time()
                model.predict(X_test)
                inf_times.append(time.time() - t0)

            results.append({
                "model": model_name, "tier": tier_name,
                "train_time_sec": round(train_time, 4),
                "inference_time_mean_sec": round(np.mean(inf_times), 6),
                "inference_time_std_sec": round(np.std(inf_times), 6),
                "n_test_samples": len(X_test),
                "needs_tuning": True,
            })
        except Exception as e:
            print(f"    {model_name} error: {e}")

    # --- Generative BGA ---
    print(f"  Timing GenerativeBGA ({tier_name})...")
    gen = GenerativeBGAModel(smoothing_alpha=1.0)
    y_tr_str = le.inverse_transform(y_train)
    t0 = time.time()
    gen.fit(X_train, y_tr_str, snp_names)
    train_time = time.time() - t0
    inf_times = []
    for _ in range(10):
        t0 = time.time()
        gen.predict(X_test)
        inf_times.append(time.time() - t0)
    results.append({
        "model": "GenerativeBGA", "tier": tier_name,
        "train_time_sec": round(train_time, 4),
        "inference_time_mean_sec": round(np.mean(inf_times), 6),
        "inference_time_std_sec": round(np.std(inf_times), 6),
        "n_test_samples": len(X_test), "needs_tuning": False,
    })

    # --- TabPFN (no tuning needed) ---
    print(f"  Timing TabPFN ({tier_name})...")
    try:
        tabpfn = make_tabpfn_classifier()
        t0 = time.time()
        tabpfn.fit(X_train, y_train)
        train_time = time.time() - t0
        inf_times = []
        for _ in range(10):
            t0 = time.time()
            tabpfn.predict(X_test)
            inf_times.append(time.time() - t0)
        results.append({
            "model": "TabPFN", "tier": tier_name,
            "train_time_sec": round(train_time, 4),
            "inference_time_mean_sec": round(np.mean(inf_times), 6),
            "inference_time_std_sec": round(np.std(inf_times), 6),
            "n_test_samples": len(X_test), "needs_tuning": False,
        })
    except Exception as e:
        print(f"    TabPFN error: {e}")

    # --- FREEFORM ---
    print(f"  Timing FREEFORM ({tier_name})...")
    freeform = FreeformBGAClassifier(top_k=15, n_interaction_pairs=45)
    t0 = time.time()
    freeform.fit(X_train, y_train)
    train_time = time.time() - t0
    inf_times = []
    for _ in range(10):
        t0 = time.time()
        freeform.predict(X_test)
        inf_times.append(time.time() - t0)
    results.append({
        "model": "FREEFORM", "tier": tier_name,
        "train_time_sec": round(train_time, 4),
        "inference_time_mean_sec": round(np.mean(inf_times), 6),
        "inference_time_std_sec": round(np.std(inf_times), 6),
        "n_test_samples": len(X_test), "needs_tuning": False,
    })

    # --- popVAE ---
    print(f"  Timing popVAE ({tier_name})...")
    pvae = PopVAEClassifier(
        latent_dim=10, enc_hidden=(128, 64), beta=1.0, gamma=10.0,
        epochs=200, patience=20, random_state=RANDOM_STATE)
    t0 = time.time()
    pvae.fit(X_train, y_train)
    train_time = time.time() - t0
    inf_times = []
    for _ in range(10):
        t0 = time.time()
        pvae.predict(X_test)
        inf_times.append(time.time() - t0)
    results.append({
        "model": "popVAE", "tier": tier_name,
        "train_time_sec": round(train_time, 4),
        "inference_time_mean_sec": round(np.mean(inf_times), 6),
        "inference_time_std_sec": round(np.std(inf_times), 6),
        "n_test_samples": len(X_test), "needs_tuning": True,
    })

    return pd.DataFrame(results)

print("=== Tier 1 (Continental) ===")
df_timing_t1 = measure_timing(X_cont, y_super, snp_ids, le_dict["super_pop"], "tier1")

print("=== Tier 2 (EAS) ===")
df_timing_t2 = measure_timing(X_eas, y_pop_eas, snp_ids_eas, le_dict_eas["pop"], "tier2")

df_timing = pd.concat([df_timing_t1, df_timing_t2], ignore_index=True)
df_timing.to_csv(OUTPUT_DIR / "timing_all_models.csv", index=False)
print("Timing summary:")
print(df_timing[["model", "tier", "train_time_sec", "inference_time_mean_sec"]].to_string(index=False))
print("Saved timing CSV.")

=== Tier 1 (Continental) ===
  Timing LogisticRegression (tier1)...
  Timing RandomForest (tier1)...
  Timing GradientBoosting (tier1)...
  Timing XGBoost (tier1)...
  Timing LightGBM (tier1)...
  Timing CatBoost (tier1)...
  Timing NGBoost (tier1)...
    NGBoost error: index 3 is out of bounds for axis 0 with size 2
  Timing GenerativeBGA (tier1)...
  Timing TabPFN (tier1)...
    TabPFN error: Failed to download TabPFN ModelVersion.V2_5 model 'tabpfn-v2.5-classifier-v2.5_default.ckpt'.

Details and instructions:
HuggingFace authentication error downloading from 'Prior-Labs/tabpfn_2_5'.
This model is gated and requires you to accept its terms.

Please follow these steps:
1. Visit https://huggingface.co/Prior-Labs/tabpfn_2_5 in your browser and accept the terms of use.
2. Log in to your Hugging Face account via the command line by running:
   hf auth login
   (Alternatively, you can set the HF_TOKEN environment variable   with a read token.)

For detailed instructions, see https://docs.

## 11. Summary

In [13]:
import glob

saved_files = sorted(OUTPUT_DIR.glob("*"))
print(f"Saved {len(saved_files)} files to {OUTPUT_DIR}:")
for f in saved_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s}  {size_kb:8.1f} KB")

print(" All analyses complete. Proceed to 02_results_visualization.ipynb.")

Saved 29 files to c:\Users\Admin\Uni\BGA-AISNP-NEW\results\analysis_arrays:
  allele_freq_tier1_top20.csv                         2.2 KB
  allele_freq_tier2_top20.csv                         1.9 KB
  benchmark_all_models.csv                            3.4 KB
  df_meta_cont.csv                                   41.6 KB
  df_meta_eas.csv                                     8.4 KB
  dimred_pca_cont.npy                                19.7 KB
  dimred_pca_cont_var.npy                             0.1 KB
  dimred_pca_eas.npy                                  4.1 KB
  dimred_pca_eas_var.npy                              0.1 KB
  dimred_popvae_cont.npy                             97.9 KB
  dimred_popvae_cont_2d.npy                          19.7 KB
  dimred_popvae_eas.npy                              19.8 KB
  dimred_popvae_eas_2d.npy                            4.1 KB
  dimred_tsne_cont.npy                               19.7 KB
  dimred_tsne_eas.npy                                 4.1 KB
  dimred_